---

## 컨텍스트 접근

도구는 에이전트 상태, 런타임 컨텍스트 및 장기 메모리에 액세스할 수 있을 때 가장 강력합니다. 이를 통해 도구는 컨텍스트 인식 결정을 내리고, 응답을 개인화하며, 대화 전반에 걸쳐 정보를 유지할 수 있습니다.

`ToolRuntime` 매개변수를 통해 다음 런타임 정보에 액세스할 수 있습니다:

| 속성 | 설명 |
|:---|:---|
| **state** | 실행을 통해 흐르는 변경 가능한 데이터 (메시지, 카운터, 커스텀 필드) |
| **context** | 사용자 ID, 세션 세부 정보 등 불변 구성 정보 |
| **store** | 대화 전반에 걸친 영구 장기 메모리 |
| **stream_writer** | 도구 실행 중 커스텀 업데이트 스트리밍 |
| **config** | 실행을 위한 RunnableConfig |
| **tool_call_id** | 현재 도구 호출의 고유 ID |

---
### 상태 업데이트 (tool_call_id)

`Command`를 사용하면 도구 내에서 에이전트의 상태를 직접 업데이트하거나 그래프의 실행 흐름을 제어할 수 있습니다. `update` 필드로 상태를 업데이트하고, `goto` 필드로 다음 노드를 지정할 수 있습니다.

상태 업데이트 시 `ToolMessage`를 포함해야 하며, `tool_call_id`는 `runtime.tool_call_id`에서 가져와야 합니다.

아래 코드는 Command를 사용하여 사용자 이름을 업데이트하는 도구 예시입니다.

In [1]:
from feature.tools.ToolRuntimeToolCallID import CustomContext, ToolRuntimeAgent
from langchain_core.messages import HumanMessage

workflow = ToolRuntimeAgent()

In [2]:
# 에이전트 실행
config = {"configurable": {"thread_id": "10"}}
_ = workflow.invoke(
    inputs={"messages": [{"role": "user", "content": "내 이름은 쥬카"}]},
    config=config,
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  update_user_name (call_XVvun5xQJZE7F9Viy15p9s9w)
 Call ID: call_XVvun5xQJZE7F9Viy15p9s9w
  Args:
    new_name: 쥬카

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
user_name:
쥬카
================================= Tool Message =================================
Name: update_user_name

Successfully updated user name to 쥬카

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

당신의 이름이 쥬카로 성공적으로 업데이트되었습니다! 어떻게 도와드릴까요?


In [3]:
_ = workflow.invoke(
    inputs={"messages": [{"role": "user", "content": "내 이름은 사실 재영이야"}]},
    config=config,
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  update_user_name (call_YfzqjQwnTPkSUI0EfQQiACs9)
 Call ID: call_YfzqjQwnTPkSUI0EfQQiACs9
  Args:
    new_name: 재영

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
user_name:
재영
================================= Tool Message =================================
Name: update_user_name

Successfully updated user name to 재영

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

당신의 이름이 재영으로 성공적으로 업데이트되었습니다! 무엇을 도와드릴까요?


In [4]:
len(workflow.graph.get_state(config).values['messages'])



8

In [5]:
_ = workflow.invoke(
    inputs={"messages": [{"role": "user", "content": "메시지 전부 삭제해줘"}]},
    config=config,
)    



🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  clear_messages (call_JDz1guH8vXSlAL5vTp7cvsDP)
 Call ID: call_JDz1guH8vXSlAL5vTp7cvsDP
  Args:

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Remove Message ================================


================================ Remove Message ================================


================================ Remove Message ================================


================================ Remove Message ================================


================================ Remove Message ================================


================================ Remove Message ================================


================================ Remove Message ================================


================================ Remove Message ================================


==========

In [6]:
len(workflow.graph.get_state(config).values['messages'])


3

In [7]:
workflow.graph.get_state(config).values['messages']


[AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 233, 'total_tokens': 243, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d682aea597', 'id': 'chatcmpl-Djb9E52zHoOHdQ0E6cMHs3I8Xi387', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e61f5-53ba-75d1-9dd7-bafb750a3582-0', tool_calls=[{'name': 'clear_messages', 'args': {}, 'id': 'call_JDz1guH8vXSlAL5vTp7cvsDP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 233, 'output_tokens': 10, 'total_tokens': 243, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 ToolMessage(content=